## load all CSV to df_sp500

In [ ]:
# get df of the sp500 cvs
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data/raw/sp500_10Y")
dfs = []
for file in DATA_DIR.glob("*.csv"):
    ticker = file.stem.upper()   # Get the ticker from the filename
    df = pd.read_csv(file, skiprows=2)
    df.columns = ["Date", "Close", "High", "Low", "Open", "Volume"]
    
    df["Ticker"] = ticker
    df["Date"] = pd.to_datetime(df["Date"])
    
    dfs.append(df)

df_sp500 = pd.concat(dfs, ignore_index=True)


## DF SP500

In [2]:
df_sp500.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1220725 entries, 0 to 1220724
Data columns (total 7 columns):
 #   Column  Non-Null Count    Dtype         
---  ------  --------------    -----         
 0   Date    1220725 non-null  datetime64[ns]
 1   Close   1220725 non-null  float64       
 2   High    1220725 non-null  float64       
 3   Low     1220725 non-null  float64       
 4   Open    1220725 non-null  float64       
 5   Volume  1220725 non-null  int64         
 6   Ticker  1220725 non-null  object        
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 65.2+ MB


In [3]:
df_sp500.head()

,Date,Close,High,Low,Open,Volume,Ticker
0,2015-12-21,37.662884,37.930386,37.303132,37.367702,1734500,A
1,2015-12-22,38.022614,38.124082,37.561398,37.828900,1643200,A
2,2015-12-23,38.529961,38.612980,38.179436,38.391595,1510700,A
3,2015-12-24,38.871262,38.981952,38.437718,38.548412,874500,A
4,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A


In [4]:
df_sp500.describe()

,Date,Close,High,Low,Open,Volume
count,1220725,1.220725e+06,1.220725e+06,1.220725e+06,1.220725e+06,1.220725e+06
mean,2021-01-11 19:02:53.738393600,1.342100e+02,1.357452e+02,1.326169e+02,1.341936e+02,6.533028e+06
min,2015-12-21 00:00:00,6.151807e-01,6.234744e-01,6.037163e-01,6.044482e-01,0.000000e+00
25%,2018-07-19 00:00:00,4.172700e+01,4.221131e+01,4.122500e+01,4.172511e+01,1.027300e+06
50%,2021-01-26 00:00:00,7.730000e+01,7.816040e+01,7.639864e+01,7.728607e+01,2.193100e+06
75%,2023-07-13 00:00:00,1.456924e+02,1.473343e+02,1.440089e+02,1.456598e+02,5.071700e+06
max,2025-12-19 00:00:00,9.924400e+03,9.964770e+03,9.794000e+03,9.914170e+03,3.692928e+09
std,NaN,2.939089e+02,2.973221e+02,2.904679e+02,2.938496e+02,2.724701e+07


## Number of stocks of sp500 and rows per ticker

In [7]:
df_sp500["Ticker"].nunique()     # number of stocks

501

In [8]:

df_sp500.groupby("Ticker").size() # group by ticker and count number of rows for each ticker


Ticker
A       2515
AAPL    2515
ABBV    2515
ABNB    1263
ABT     2515
        ... 
XYZ     2515
YUM     2515
ZBH     2515
ZBRA    2515
ZTS     2515
Length: 501, dtype: int64

In [9]:
df = df_sp500.sort_values(["Ticker", "Date"]).copy()

## Create target y

In [ ]:
# group(ticker) ensures the computation is performed separately for each stock
# shift(-1) moves the closing price one day forward, giving the next trading day’s closing price.
# Dividing the next-day price by today’s price and subtracting 1 produces the percentage change between the two days
df["y_return"] = (
    df.groupby("Ticker")["Close"].shift(-1) / df["Close"] - 1
)


## Classification target

In [ ]:
df["y_direction"] = (df["y_return"] > 0).astype(int) # 

## Return based features 

In [12]:
import numpy as np
df["Returnlag"] = df.groupby("Ticker")["Close"].pct_change()
df["log_return"] = (
    df.groupby("Ticker")["Close"]
      .transform(lambda x: np.log(x).diff())
)


## Feature engineering

In [13]:
df["HL_range"] = (df["High"] - df["Low"]) / df["Close"]
df["OC_change"] = (df["Close"] - df["Open"]) / df["Open"]

df["Volume_log"] = np.log1p(df["Volume"])

df["MA_5"] = df.groupby("Ticker")["Close"].transform(
    lambda x: x.rolling(5).mean()
)
df["Volatility_5"] = df.groupby("Ticker")["y_return"].transform(
    lambda x: x.rolling(5).std()
)

df = df.dropna()
df.groupby("Ticker").size()


Ticker
A       2510
AAPL    2510
ABBV    2510
ABNB    1258
ABT     2510
        ... 
XYZ     2510
YUM     2510
ZBH     2510
ZBRA    2510
ZTS     2510
Length: 501, dtype: int64

## Define x and y

In [14]:
FEATURES = [
    "Open", "High", "Low", "Close",
    "HL_range", "OC_change",
    "Volume_log", "MA_5", "Volatility_5"
]

X = df[FEATURES]
y = df["y_return"]


## Split into train and test based on date

In [15]:
split_date = df["Date"].quantile(0.8)

train_idx = df["Date"] <= split_date
test_idx  = df["Date"] > split_date

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]



## Save new sp500 dataset

In [16]:
df.to_csv("data/processed/sp500_panel.csv", index=False)


In [17]:
df.head()

,Date,Close,High,Low,Open,Volume,Ticker,y_return,y_direction,Returnlag,log_return,HL_range,OC_change,Volume_log,MA_5,Volatility_5
4,2015-12-28,38.539181,38.815913,38.299350,38.760566,1458200,A,0.013882,1,-0.008543,-0.008580,0.013404,-0.005712,14.192714,38.325180,0.009196
5,2015-12-29,39.074196,39.184887,38.668324,38.815916,1757000,A,-0.004485,0,0.013882,0.013787,0.013220,0.006654,14.379119,38.607442,0.010441
6,2015-12-30,38.898933,39.092647,38.751345,39.074198,834300,A,-0.005826,0,-0.004485,-0.004495,0.008774,-0.004485,13.634350,38.782706,0.009940
7,2015-12-31,38.672314,39.171786,38.589068,38.755560,1451000,A,-0.026788,0,-0.005826,-0.005843,0.015068,-0.002148,14.187764,38.811177,0.014453
8,2016-01-04,37.636372,38.098849,37.312639,37.978607,3287300,A,-0.003441,0,-0.026788,-0.027153,0.020890,-0.009011,15.005577,38.564199,0.014440
